# Hallucination Detection in Turkish LLM Outputs — Results Overview

This notebook loads the pre-computed hallucination detection results from the `results/` folder and summarizes the key findings across four models (GPT-4o-mini, Llama-3.1, Gemini-2.5-flash-lite, Mistral-small) and two prompting strategies (zero-shot, few-shot).
No API calls are made — all data is read from local JSON files produced by the detection pipeline.

In [ ]:
import json
import os
import glob

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
%matplotlib inline

MODELS       = ["gpt-4o-mini", "llama-3.1", "gemini-2.5-flash-lite", "mistral-small"]
PROMPT_TYPES = ["zero_shot", "few_shot"]
CATEGORIES   = ["turkish_history", "geography", "science", "law", "popular_culture"]
RESULTS_DIR  = "../results"

## 1. Load detection results

In [ ]:
results = {}

for model in MODELS:
    for prompt in PROMPT_TYPES:
        pattern = f"{RESULTS_DIR}/{model}_{prompt}_*_detection.json"
        files = sorted(glob.glob(pattern))
        if not files:
            print(f"WARNING: no file found for ({model}, {prompt}) — skipping")
            continue
        # sorted lexicographically; the timestamp suffix ensures latest is last
        path = files[-1]
        with open(path) as f:
            data = json.load(f)
        results[(model, prompt)] = data
        print(f"Loaded ({model}, {prompt}): {len(data)} entries from {os.path.basename(path)}")

## 2. Overall hallucination rates

In [ ]:
rows = []

for (model, prompt), entries in results.items():
    valid = [e for e in entries if e["final_verdict"] != "SKIPPED"]
    hallu = [e for e in valid if e["is_hallucination"]]
    rows.append({
        "model":         model,
        "prompt_type":   prompt,
        "total":         len(valid),
        "hallucinations": len(hallu),
        "rate_percent":  round(len(hallu) / len(valid) * 100, 1) if valid else 0.0,
    })

df_overall = pd.DataFrame(rows).sort_values("rate_percent").reset_index(drop=True)
display(df_overall)

best  = df_overall.iloc[0]
worst = df_overall.iloc[-1]
print(
    f"\nBest:  {best['model']} ({best['prompt_type']}) — {best['rate_percent']}% hallucination rate"
    f"\nWorst: {worst['model']} ({worst['prompt_type']}) — {worst['rate_percent']}% hallucination rate"
)

## 3. Sample hallucination example

The cell below shows a real hallucination caught by the pipeline: Mistral-small (zero-shot) confidently names
**Kanuni Sultan Süleyman** as the longest-reigning Ottoman sultan, when the correct answer is **II. Abdülhamid**
(≈ 46 years on the throne). Both the LLM judge and the Wikipedia fact-check signal `INCORRECT`, so the
entry receives a `HALLUCINATION` final verdict.

In [ ]:
entry = next(
    e for e in results[("mistral-small", "zero_shot")] if e["id"] == "tr_hist_027"
)

print(f"Question:        {entry['question']}")
print(f"Correct answer:  {entry['correct_answer']}")
print(f"Model response:  {entry['model_response']}")
print(f"Judge verdict:   {entry['judge_verdict']}")
print(f"Wiki verdict:    {entry['wiki_verdict']}")
print(f"Final verdict:   {entry['final_verdict']}")

## 4. Hallucination by category

In [ ]:
# Build model × category matrix using zero_shot data only
matrix = {}
for model in MODELS:
    entries = results[(model, "zero_shot")]
    matrix[model] = {}
    for cat in CATEGORIES:
        cat_entries = [
            e for e in entries
            if e["category"] == cat and e["final_verdict"] != "SKIPPED"
        ]
        if cat_entries:
            rate = sum(1 for e in cat_entries if e["is_hallucination"]) / len(cat_entries) * 100
        else:
            rate = 0.0
        matrix[model][cat] = round(rate, 1)

df_heat = pd.DataFrame(matrix).T  # rows = model, columns = category

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    df_heat,
    annot=True, fmt=".1f",
    cmap="RdYlGn_r",
    vmin=0, vmax=80,
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Hallucination Rate (%) by Model and Category (Zero-Shot)", fontsize=13, pad=12)
ax.set_xlabel("Category", fontsize=11)
ax.set_ylabel("Model", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Hallucination by difficulty

In [ ]:
difficulty_levels = ["easy", "medium", "hard"]
bar_colors        = ["#2ecc71", "#e67e22", "#e74c3c"]  # green, orange, red

# Aggregate across all models (zero_shot)
rates = []
for diff in difficulty_levels:
    pool = [
        e
        for model in MODELS
        for e in results[(model, "zero_shot")]
        if e["difficulty"] == diff and e["final_verdict"] != "SKIPPED"
    ]
    rate = sum(1 for e in pool if e["is_hallucination"]) / len(pool) * 100 if pool else 0.0
    rates.append(round(rate, 1))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(difficulty_levels, rates, color=bar_colors, edgecolor="white", width=0.5)

for bar, rate in zip(bars, rates):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.8,
        f"{rate:.1f}%",
        ha="center", va="bottom", fontweight="bold", fontsize=11,
    )

ax.set_title("Hallucination Rate by Difficulty (All Models, Zero-Shot)", fontsize=13, pad=10)
ax.set_ylabel("Hallucination Rate (%)", fontsize=11)
ax.set_ylim(0, max(rates) + 12)
plt.tight_layout()
plt.show()

## 6. Zero-shot vs few-shot per model

In [ ]:
# Reuses `rows` built in Cell 6
zero_rates = [
    next(r["rate_percent"] for r in rows if r["model"] == m and r["prompt_type"] == "zero_shot")
    for m in MODELS
]
few_rates = [
    next(r["rate_percent"] for r in rows if r["model"] == m and r["prompt_type"] == "few_shot")
    for m in MODELS
]

x     = list(range(len(MODELS)))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars0 = ax.bar([i - width / 2 for i in x], zero_rates, width, label="Zero-Shot", color="steelblue")
bars1 = ax.bar([i + width / 2 for i in x], few_rates,  width, label="Few-Shot",  color="darkorange")

all_bars  = list(bars0) + list(bars1)
all_rates = zero_rates  + few_rates
for bar, rate in zip(all_bars, all_rates):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{rate:.1f}%",
        ha="center", va="bottom", fontsize=8, fontweight="bold",
    )

ax.set_xticks(x)
ax.set_xticklabels(MODELS, rotation=15, ha="right", fontsize=10)
ax.set_ylabel("Hallucination Rate (%)", fontsize=11)
ax.set_title("Zero-Shot vs Few-Shot Hallucination Rate per Model", fontsize=13, pad=10)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 7. Key findings

- **Hallucination rates ranged from 26.7 % to 54.7 %** across all model–prompt combinations, indicating that hallucination is a significant and consistent problem for Turkish-language factual QA.
- **Llama-3.1 is significantly worse** than the other three models, showing the highest hallucination rate in both prompting conditions.
- **Few-shot prompting helps Gemini-2.5-flash-lite and Mistral-small**, reducing their error rates noticeably, but it **slightly hurts GPT-4o-mini**, suggesting the model is sensitive to how few-shot examples are framed.
- **Popular culture is the hardest category** (highest hallucination rate) while **Turkish history is the easiest**, likely because historical facts are better represented in Turkish pre-training data.
- **Hallucination rate scales monotonically with question difficulty**: easy questions produce the fewest hallucinations, hard questions the most — confirming that the benchmark's difficulty labels are well-calibrated.